In [ ]:
!pip install zarr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.0 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import time
import shutil
import zipfile
import tempfile
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
import gcsfs
import zarr
import xarray as xr

from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore")

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("zarr:", zarr.__version__)
print("xarray:", xr.__version__)

numpy: 2.0.2
pandas: 2.2.2
zarr: 3.2.1
xarray: 2025.12.0


In [ ]:
YEARS = list(range(2015, 2025))
MONTHS = list(range(1, 13))

DRIVE_ROOT = "/content/drive/MyDrive/aviation_weather_disruption_us_10y"
LOCAL_WORK_DIR = "/content/aviation_weather_work"

ZARR_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# Start stable. Increase later only after one month works.
POINT_BATCH_SIZE = 3000
AIRPORT_SHARD_SIZE = 5000
MAX_ARCO_VAR_WORKERS = 2

SKIP_EXISTING = True
WRITE_SUCCESS_MARKERS = True

# Do BTS separately after ARCO starts working.
DOWNLOAD_BTS = False

In [ ]:
drive.mount("/content/drive")

DIRS = {
    "root": DRIVE_ROOT,
    "ourairports": os.path.join(DRIVE_ROOT, "metadata", "ourairports"),
    "manifests": os.path.join(DRIVE_ROOT, "metadata", "project_manifests"),
    "arco_weather": os.path.join(DRIVE_ROOT, "bronze", "arco_era5_us_airport_hourly"),
    "bts_raw": os.path.join(DRIVE_ROOT, "bronze", "bts_on_time", "raw_zip"),
    "bts_parquet": os.path.join(DRIVE_ROOT, "bronze", "bts_on_time", "parquet"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
OURAIRPORTS_FILES = {
    "airports": "https://davidmegginson.github.io/ourairports-data/airports.csv",
    "runways": "https://davidmegginson.github.io/ourairports-data/runways.csv",
    "airport_frequencies": "https://davidmegginson.github.io/ourairports-data/airport-frequencies.csv",
    "countries": "https://davidmegginson.github.io/ourairports-data/countries.csv",
    "regions": "https://davidmegginson.github.io/ourairports-data/regions.csv",
}

def download_file(url, out_path, retries=3):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print("SKIP existing:", out_path)
        return out_path

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    headers = {"User-Agent": "Mozilla/5.0"}

    for attempt in range(1, retries + 1):
        try:
            tmp = out_path + ".part"

            with requests.get(url, stream=True, timeout=(30, 300), headers=headers) as r:
                r.raise_for_status()

                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            os.replace(tmp, out_path)
            print("Downloaded:", out_path)
            return out_path

        except Exception as e:
            print(f"Attempt {attempt}/{retries} failed:", e)
            time.sleep(3 * attempt)

    raise RuntimeError(f"Failed to download: {url}")


for name, url in OURAIRPORTS_FILES.items():
    csv_path = os.path.join(DIRS["ourairports"], f"{name}.csv")
    parquet_path = os.path.join(DIRS["ourairports"], f"{name}.parquet")

    download_file(url, csv_path)

    if not os.path.exists(parquet_path):
        df_tmp = pd.read_csv(csv_path, low_memory=False)
        df_tmp.to_parquet(parquet_path, index=False, compression="zstd")
        print(f"Wrote parquet: {parquet_path}, rows={len(df_tmp):,}")

SKIP existing: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/airports.csv
SKIP existing: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/runways.csv
SKIP existing: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/airport_frequencies.csv
SKIP existing: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/countries.csv
SKIP existing: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/regions.csv


In [ ]:
airports_raw = pd.read_csv(
    os.path.join(DIRS["ourairports"], "airports.csv"),
    low_memory=False
)

airports_raw = airports_raw.dropna(subset=["latitude_deg", "longitude_deg"]).copy()

for c in airports_raw.columns:
    if airports_raw[c].dtype == "object":
        airports_raw[c] = airports_raw[c].astype("string")

us = airports_raw[airports_raw["iso_country"] == "US"].copy()

# IMPORTANT FIX
us_non_closed = us[~us["type"].isin(["closed", "closed_airport"])].copy()

# Maximum US scope but remove closed facilities
airports = us_non_closed.copy().reset_index(drop=True)
airports["airport_key"] = np.arange(len(airports), dtype=np.int32)

print("Selected airport count:", len(airports))
print(airports["type"].value_counts(dropna=False))

airport_dim_selected_path = os.path.join(
    DIRS["ourairports"],
    "us_airports_selected.parquet"
)

airports.to_parquet(
    airport_dim_selected_path,
    index=False,
    compression="zstd"
)

print("Saved:", airport_dim_selected_path)

Selected airport count: 25126
type
small_airport     15285
heliport           8218
medium_airport      821
seaplane_base       675
large_airport        95
balloonport          32
Name: count, dtype: Int64
Saved: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/us_airports_selected.parquet


In [ ]:
airports_csv_path = os.path.join(DIRS["ourairports"], "airports.csv")
airports_parquet_path = os.path.join(DIRS["ourairports"], "airports.parquet")

if os.path.exists(airports_parquet_path):
    airports_raw = pd.read_parquet(airports_parquet_path)
else:
    airports_raw = pd.read_csv(airports_csv_path, low_memory=False)

airports_raw = airports_raw.dropna(subset=["latitude_deg", "longitude_deg"]).copy()

for c in airports_raw.columns:
    if airports_raw[c].dtype == "object":
        airports_raw[c] = airports_raw[c].astype("string")

us = airports_raw[airports_raw["iso_country"] == "US"].copy()

# IMPORTANT: OurAirports uses "closed", not always "closed_airport"
us_non_closed = us[~us["type"].isin(["closed", "closed_airport"])].copy()

airports = us_non_closed.copy().reset_index(drop=True)
airports["airport_key"] = np.arange(len(airports), dtype=np.int32)

print("US total with coordinates:", len(us))
print("Selected non-closed US airport/facility count:", len(airports))
print(airports["type"].value_counts(dropna=False))

airport_dim_selected_path = os.path.join(
    DIRS["ourairports"],
    "us_airports_selected.parquet"
)

airports.to_parquet(
    airport_dim_selected_path,
    index=False,
    compression="zstd"
)

print("Saved:", airport_dim_selected_path)

US total with coordinates: 32485
Selected non-closed US airport/facility count: 25126
type
small_airport     15285
heliport           8218
medium_airport      821
seaplane_base       675
large_airport        95
balloonport          32
Name: count, dtype: Int64
Saved: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/us_airports_selected.parquet


In [ ]:
# ============================================================
# CELL 1: OPEN ARCO-ERA5 AND CHECK VARIABLES
# ============================================================

ARCO_VARIABLES = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "10m_wind_gust_since_previous_post_processing",
    "surface_pressure",
    "mean_sea_level_pressure",
    "total_precipitation",
    "total_cloud_cover",
    "convective_available_potential_energy",
]

print("Opening ARCO-ERA5 Zarr...")

fs = gcsfs.GCSFileSystem(token="anon")
mapper = fs.get_mapper(ZARR_PATH)

root = zarr.open_group(mapper, mode="r")

ds_meta = xr.open_zarr(
    ZARR_PATH,
    chunks=None,
    storage_options={"token": "anon"},
)

lats = np.asarray(root["latitude"][:])
lons = np.asarray(root["longitude"][:])
times = pd.DatetimeIndex(ds_meta["time"].values)

print("ARCO time range:", times[0], "to", times[-1])
print("Latitude count:", len(lats))
print("Longitude count:", len(lons))

available_vars = set(root.array_keys())
missing = [v for v in ARCO_VARIABLES if v not in available_vars]

if missing:
    print("Missing variables:", missing)
    print("Available sample:", sorted(list(available_vars))[:120])
    raise ValueError("Fix ARCO_VARIABLES before continuing.")

print("All selected ARCO variables exist.")

Opening ARCO-ERA5 Zarr...
ARCO time range: 1900-01-01 00:00:00 to 2050-12-31 23:00:00
Latitude count: 721
Longitude count: 1440
All selected ARCO variables exist.


In [ ]:
# ============================================================
# CELL 2: MAP US AIRPORTS TO NEAREST ERA5 GRID CELLS
# ============================================================

airport_dim_selected_path = os.path.join(
    DIRS["ourairports"],
    "us_airports_selected.parquet"
)

airports = pd.read_parquet(airport_dim_selected_path)

airport_lats = airports["latitude_deg"].to_numpy(dtype=np.float64)
airport_lons_raw = airports["longitude_deg"].to_numpy(dtype=np.float64)

# Match ARCO longitude convention
if np.nanmin(lons) >= 0 and np.nanmax(lons) > 180:
    airport_lons = airport_lons_raw % 360.0
else:
    airport_lons = ((airport_lons_raw + 180.0) % 360.0) - 180.0

lat_idx = np.array(
    [int(np.abs(lats - x).argmin()) for x in airport_lats],
    dtype=np.int32
)

lon_idx = np.array(
    [int(np.abs(lons - x).argmin()) for x in airport_lons],
    dtype=np.int32
)

airports["era5_lat_idx"] = lat_idx
airports["era5_lon_idx"] = lon_idx
airports["era5_grid_lat"] = lats[lat_idx]
airports["era5_grid_lon"] = lons[lon_idx]

point_pairs = np.column_stack([lat_idx, lon_idx]).astype(np.int32)

unique_pairs, point_inverse = np.unique(
    point_pairs,
    axis=0,
    return_inverse=True,
)

unique_lat_idx = unique_pairs[:, 0].astype(np.int64)
unique_lon_idx = unique_pairs[:, 1].astype(np.int64)
point_inverse = point_inverse.astype(np.int32)

airports["era5_point_key"] = point_inverse

airport_dim_with_grid_path = os.path.join(
    DIRS["ourairports"],
    "us_airports_selected_with_era5_grid.parquet"
)

airports.to_parquet(
    airport_dim_with_grid_path,
    index=False,
    compression="zstd"
)

print("Selected US airport/facility rows:", len(airports))
print("Unique ERA5 grid cells:", len(unique_pairs))
print("Saved:", airport_dim_with_grid_path)

display(airports[[
    "airport_key", "ident", "type", "name",
    "latitude_deg", "longitude_deg",
    "era5_grid_lat", "era5_grid_lon",
    "era5_point_key"
]].head())

Selected US airport/facility rows: 25126
Unique ERA5 grid cells: 8807
Saved: /content/drive/MyDrive/aviation_weather_disruption_us_10y/metadata/ourairports/us_airports_selected_with_era5_grid.parquet


,airport_key,ident,type,name,latitude_deg,longitude_deg,era5_grid_lat,era5_grid_lon,era5_point_key
0,0,00A,heliport,Total RF Heliport,40.070985,-74.933689,40.00,285.00,4106
1,1,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,38.75,258.50,4684
2,2,00AK,small_airport,Lowell Field,59.947733,-151.692524,60.00,208.25,339
3,3,00AL,small_airport,Epps Airpark,34.864799,-86.770302,34.75,273.25,6594
4,4,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,59.00,203.50,392


In [ ]:
# ============================================================
# CELL 3: HELPERS
# ============================================================

def get_month_time_bounds(year: int, month: int):
    start = pd.Timestamp(year=year, month=month, day=1)

    if month == 12:
        end = pd.Timestamp(year=year + 1, month=1, day=1)
    else:
        end = pd.Timestamp(year=year, month=month + 1, day=1)

    t0 = int(np.searchsorted(times.values, np.datetime64(start)))
    t1 = int(np.searchsorted(times.values, np.datetime64(end)))

    selected = times[t0:t1]

    if len(selected) == 0:
        raise ValueError(f"No ARCO times found for {year}-{month:02d}")

    return t0, t1, selected


def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"

    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)

    os.replace(tmp, path)


def atomic_copy(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".tmp"
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


def arco_month_success_path(year, month):
    return os.path.join(
        DIRS["arco_weather"],
        f"year={year}",
        f"month={month:02d}",
        "_SUCCESS.json"
    )


def arco_month_done(year, month):
    return SKIP_EXISTING and os.path.exists(arco_month_success_path(year, month))


# Quick sanity check
test_t0, test_t1, test_times = get_month_time_bounds(2015, 1)
print(test_t0, test_t1)
print(test_times[0], "to", test_times[-1])
print("hours:", len(test_times))

1008072 1008816
2015-01-01 00:00:00 to 2015-01-31 23:00:00
hours: 744


In [ ]:
# ============================================================
# CELL 4: SMALL READ TEST
# ============================================================

year, month = 2015, 1
global_t0, global_t1, month_times = get_month_time_bounds(year, month)

n_hours = len(month_times)
test_points = min(100, len(unique_pairs))

arr = root["2m_temperature"]

time_index = np.arange(global_t0, global_t1, dtype=np.int64)

lat_batch = unique_lat_idx[:test_points]
lon_batch = unique_lon_idx[:test_points]

tt = np.broadcast_to(time_index[:, None], (n_hours, test_points))
yy = np.broadcast_to(lat_batch[None, :], (n_hours, test_points))
xx = np.broadcast_to(lon_batch[None, :], (n_hours, test_points))

try:
    vals = arr.vindex[tt, yy, xx]
except Exception:
    vals = arr.get_coordinate_selection((tt, yy, xx))

vals = np.asarray(vals, dtype=np.float32)

print("Shape:", vals.shape)
print("Min:", np.nanmin(vals))
print("Max:", np.nanmax(vals))

del vals, tt, yy, xx
gc.collect()

Shape: (744, 100)
Min: 222.1852
Max: 273.92294


0

In [ ]:
# ============================================================
# CELL 5: VARIABLE READER TO MEMMAP
# ============================================================

def read_arco_var_unique_points_to_memmap(
    var_name,
    global_t0,
    global_t1,
    n_hours,
    n_points,
    month_tmp_dir,
):
    arr = root[var_name]

    if arr.ndim != 3:
        raise ValueError(f"{var_name} has ndim={arr.ndim}, expected 3")

    mmap_path = os.path.join(month_tmp_dir, f"{var_name}.float32.mmap")

    out = np.memmap(
        mmap_path,
        dtype="float32",
        mode="w+",
        shape=(n_hours, n_points),
    )

    time_index = np.arange(global_t0, global_t1, dtype=np.int64)
    fill_value = arr.attrs.get("_FillValue", None)

    for p0 in tqdm(
        range(0, n_points, POINT_BATCH_SIZE),
        desc=f"Reading {var_name}",
        leave=False,
    ):
        p1 = min(p0 + POINT_BATCH_SIZE, n_points)

        lat_batch = unique_lat_idx[p0:p1]
        lon_batch = unique_lon_idx[p0:p1]
        n_batch = p1 - p0

        tt = np.broadcast_to(time_index[:, None], (n_hours, n_batch))
        yy = np.broadcast_to(lat_batch[None, :], (n_hours, n_batch))
        xx = np.broadcast_to(lon_batch[None, :], (n_hours, n_batch))

        try:
            vals = arr.vindex[tt, yy, xx]
        except Exception:
            vals = arr.get_coordinate_selection((tt, yy, xx))

        vals = np.asarray(vals, dtype=np.float32)

        if fill_value is not None:
            vals[vals == fill_value] = np.nan

        out[:, p0:p1] = vals
        out.flush()

        del vals, tt, yy, xx
        gc.collect()

    return var_name, mmap_path

In [ ]:
# ============================================================
# CELL 6: WRITE MONTHLY AIRPORT SHARDS TO PARQUET
# ============================================================

def write_arco_month_shards_from_memmaps(
    year,
    month,
    month_times,
    var_mmaps,
    n_unique_points,
):
    n_hours = len(month_times)
    n_airports = len(airports)

    airport_keys = airports["airport_key"].to_numpy(dtype=np.int32)
    inv = point_inverse

    time_values = month_times.values.astype("datetime64[ns]")
    day_values = month_times.day.to_numpy(dtype=np.int16)
    hour_values = month_times.hour.to_numpy(dtype=np.int8)

    written_files = []

    for shard_id, s0 in enumerate(
        tqdm(
            range(0, n_airports, AIRPORT_SHARD_SIZE),
            desc=f"Writing ARCO {year}-{month:02d} shards",
            leave=False,
        )
    ):
        s1 = min(s0 + AIRPORT_SHARD_SIZE, n_airports)
        shard_airport_count = s1 - s0

        shard_inv = inv[s0:s1]
        shard_airport_keys = airport_keys[s0:s1]

        arrays = {
            "time_utc": pa.array(np.repeat(time_values, shard_airport_count)),
            "airport_key": pa.array(np.tile(shard_airport_keys, n_hours), type=pa.int32()),
            "day": pa.array(np.repeat(day_values, shard_airport_count), type=pa.int16()),
            "hour_utc": pa.array(np.repeat(hour_values, shard_airport_count), type=pa.int8()),
        }

        for var_name, mmap_path in var_mmaps.items():
            mm = np.memmap(
                mmap_path,
                dtype="float32",
                mode="r",
                shape=(n_hours, n_unique_points),
            )

            vals = np.asarray(mm[:, shard_inv], dtype=np.float32).reshape(-1)
            arrays[var_name] = pa.array(vals, type=pa.float32())

            del mm, vals
            gc.collect()

        table = pa.table(arrays)

        local_part_dir = os.path.join(
            LOCAL_WORK_DIR,
            "arco_era5_us_airport_hourly",
            f"year={year}",
            f"month={month:02d}",
            f"airport_shard={shard_id:04d}",
        )

        drive_part_dir = os.path.join(
            DIRS["arco_weather"],
            f"year={year}",
            f"month={month:02d}",
            f"airport_shard={shard_id:04d}",
        )

        os.makedirs(local_part_dir, exist_ok=True)
        os.makedirs(drive_part_dir, exist_ok=True)

        local_file = os.path.join(local_part_dir, "part-00000.parquet")
        drive_file = os.path.join(drive_part_dir, "part-00000.parquet")

        pq.write_table(
            table,
            local_file,
            compression="zstd",
            compression_level=3,
            use_dictionary=True,
            row_group_size=250_000,
        )

        atomic_copy(local_file, drive_file)
        written_files.append(drive_file)

        del table, arrays
        gc.collect()

    return written_files

In [ ]:
# ============================================================
# CELL 7: TEST ONE MONTH ONLY
# ============================================================

TEST_YEAR = 2015
TEST_MONTH = 1

if arco_month_done(TEST_YEAR, TEST_MONTH):
    print(f"SKIP existing ARCO month: {TEST_YEAR}-{TEST_MONTH:02d}")
else:
    print(f"Processing test month: {TEST_YEAR}-{TEST_MONTH:02d}")

    month_start = time.time()

    global_t0, global_t1, month_times = get_month_time_bounds(TEST_YEAR, TEST_MONTH)

    n_hours = len(month_times)
    n_unique_points = len(unique_pairs)
    n_airports = len(airports)

    print("Hours:", n_hours)
    print("Airports:", n_airports)
    print("Unique ERA5 grid cells:", n_unique_points)
    print("Expected rows:", n_hours * n_airports)

    month_tmp_dir = tempfile.mkdtemp(
        prefix=f"arco_us_{TEST_YEAR}_{TEST_MONTH:02d}_",
        dir="/content",
    )

    var_mmaps = {}

    try:
        with ThreadPoolExecutor(max_workers=MAX_ARCO_VAR_WORKERS) as executor:
            futures = [
                executor.submit(
                    read_arco_var_unique_points_to_memmap,
                    var,
                    global_t0,
                    global_t1,
                    n_hours,
                    n_unique_points,
                    month_tmp_dir,
                )
                for var in ARCO_VARIABLES
            ]

            for fut in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"Reading ARCO vars {TEST_YEAR}-{TEST_MONTH:02d}",
            ):
                var_name, mmap_path = fut.result()
                var_mmaps[var_name] = mmap_path
                print("Read complete:", var_name)

        written_files = write_arco_month_shards_from_memmaps(
            year=TEST_YEAR,
            month=TEST_MONTH,
            month_times=month_times,
            var_mmaps=var_mmaps,
            n_unique_points=n_unique_points,
        )

        elapsed = time.time() - month_start

        month_manifest = {
            "year": int(TEST_YEAR),
            "month": int(TEST_MONTH),
            "hours": int(n_hours),
            "airport_count": int(n_airports),
            "unique_era5_grid_cell_count": int(n_unique_points),
            "rows": int(n_hours * n_airports),
            "variables": ARCO_VARIABLES,
            "files": written_files,
            "file_count": int(len(written_files)),
            "elapsed_seconds": round(elapsed, 2),
        }

        write_json(arco_month_success_path(TEST_YEAR, TEST_MONTH), month_manifest)

        write_json(
            os.path.join(
                DIRS["manifests"],
                f"arco_weather_manifest_year={TEST_YEAR}_month={TEST_MONTH:02d}.json",
            ),
            month_manifest,
        )

        print("Finished test month.")
        print("Rows:", f"{month_manifest['rows']:,}")
        print("Files:", len(written_files))
        print("Elapsed minutes:", round(elapsed / 60, 2))

    finally:
        shutil.rmtree(month_tmp_dir, ignore_errors=True)
        gc.collect()

SKIP existing ARCO month: 2015-01


In [ ]:
# ============================================================
# CELL 8: FULL 10-YEAR ARCO EXTRACTION
# ============================================================

arco_overall_manifest = {
    "zarr_path": ZARR_PATH,
    "years": YEARS,
    "months": MONTHS,
    "airport_count": int(len(airports)),
    "unique_era5_grid_cell_count": int(len(unique_pairs)),
    "variables": ARCO_VARIABLES,
    "airport_dim_with_grid_path": airport_dim_with_grid_path,
    "partitions": [],
}

start_all = time.time()

for year in YEARS:
    for month in MONTHS:
        success_path = arco_month_success_path(year, month)

        if arco_month_done(year, month):
            print(f"SKIP existing ARCO month: {year}-{month:02d}")

            try:
                with open(success_path, "r") as f:
                    arco_overall_manifest["partitions"].append(json.load(f))
            except Exception:
                pass

            continue

        print("\n" + "-" * 90)
        print(f"Processing ARCO {year}-{month:02d}")
        print("-" * 90)

        month_start = time.time()

        global_t0, global_t1, month_times = get_month_time_bounds(year, month)

        n_hours = len(month_times)
        n_unique_points = len(unique_pairs)
        n_airports = len(airports)

        print("Hours:", n_hours)
        print("Airports:", n_airports)
        print("Unique ERA5 grid cells:", n_unique_points)
        print("Expected rows:", f"{n_hours * n_airports:,}")

        month_tmp_dir = tempfile.mkdtemp(
            prefix=f"arco_us_{year}_{month:02d}_",
            dir="/content",
        )

        var_mmaps = {}

        try:
            with ThreadPoolExecutor(max_workers=MAX_ARCO_VAR_WORKERS) as executor:
                futures = [
                    executor.submit(
                        read_arco_var_unique_points_to_memmap,
                        var,
                        global_t0,
                        global_t1,
                        n_hours,
                        n_unique_points,
                        month_tmp_dir,
                    )
                    for var in ARCO_VARIABLES
                ]

                for fut in tqdm(
                    as_completed(futures),
                    total=len(futures),
                    desc=f"Reading ARCO vars {year}-{month:02d}",
                ):
                    var_name, mmap_path = fut.result()
                    var_mmaps[var_name] = mmap_path
                    print("Read complete:", var_name)

            written_files = write_arco_month_shards_from_memmaps(
                year=year,
                month=month,
                month_times=month_times,
                var_mmaps=var_mmaps,
                n_unique_points=n_unique_points,
            )

            elapsed = time.time() - month_start

            month_manifest = {
                "year": int(year),
                "month": int(month),
                "hours": int(n_hours),
                "airport_count": int(n_airports),
                "unique_era5_grid_cell_count": int(n_unique_points),
                "rows": int(n_hours * n_airports),
                "variables": ARCO_VARIABLES,
                "files": written_files,
                "file_count": int(len(written_files)),
                "elapsed_seconds": round(elapsed, 2),
            }

            write_json(success_path, month_manifest)

            write_json(
                os.path.join(
                    DIRS["manifests"],
                    f"arco_weather_manifest_year={year}_month={month:02d}.json",
                ),
                month_manifest,
            )

            arco_overall_manifest["partitions"].append(month_manifest)

            print(f"Finished ARCO {year}-{month:02d}")
            print("Rows:", f"{month_manifest['rows']:,}")
            print("Files:", len(written_files))
            print("Elapsed minutes:", round(elapsed / 60, 2))

        finally:
            shutil.rmtree(month_tmp_dir, ignore_errors=True)
            gc.collect()

arco_overall_manifest["elapsed_total_seconds"] = round(time.time() - start_all, 2)

write_json(
    os.path.join(
        DIRS["manifests"],
        "arco_era5_us_airport_hourly_manifest_overall.json",
    ),
    arco_overall_manifest,
)

print("\nDONE")
print("Drive ARCO output:", DIRS["arco_weather"])
print("Overall manifest:", os.path.join(DIRS["manifests"], "arco_era5_us_airport_hourly_manifest_overall.json"))

SKIP existing ARCO month: 2015-01
SKIP existing ARCO month: 2015-02
SKIP existing ARCO month: 2015-03
SKIP existing ARCO month: 2015-04
SKIP existing ARCO month: 2015-05
SKIP existing ARCO month: 2015-06
SKIP existing ARCO month: 2015-07
SKIP existing ARCO month: 2015-08
SKIP existing ARCO month: 2015-09
SKIP existing ARCO month: 2015-10
SKIP existing ARCO month: 2015-11
SKIP existing ARCO month: 2015-12
SKIP existing ARCO month: 2016-01
SKIP existing ARCO month: 2016-02
SKIP existing ARCO month: 2016-03
SKIP existing ARCO month: 2016-04
SKIP existing ARCO month: 2016-05
SKIP existing ARCO month: 2016-06
SKIP existing ARCO month: 2016-07
SKIP existing ARCO month: 2016-08
SKIP existing ARCO month: 2016-09
SKIP existing ARCO month: 2016-10
SKIP existing ARCO month: 2016-11
SKIP existing ARCO month: 2016-12
SKIP existing ARCO month: 2017-01
SKIP existing ARCO month: 2017-02
SKIP existing ARCO month: 2017-03
SKIP existing ARCO month: 2017-04
SKIP existing ARCO month: 2017-05
SKIP existing 

In [ ]:
import os
import json
import math
import pandas as pd
import pyarrow.parquet as pq

# ---------------------------------------------------------------------
# Verification config
# ---------------------------------------------------------------------

EXPECTED_YEARS = list(range(2015, 2025))
EXPECTED_MONTHS = list(range(1, 13))

EXPECTED_VARIABLES = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "10m_wind_gust_since_previous_post_processing",
    "surface_pressure",
    "mean_sea_level_pressure",
    "total_precipitation",
    "total_cloud_cover",
    "convective_available_potential_energy",
]

ARCO_ROOT = DIRS["arco_weather"]
MANIFEST_ROOT = DIRS["manifests"]

airport_count = len(airports)
expected_shards_per_month = math.ceil(airport_count / AIRPORT_SHARD_SIZE)

print("ARCO root:", ARCO_ROOT)
print("Airport count:", airport_count)
print("Expected shards per month:", expected_shards_per_month)
print("Expected month partitions:", len(EXPECTED_YEARS) * len(EXPECTED_MONTHS))

ARCO root: /content/drive/MyDrive/aviation_weather_disruption_us_10y/bronze/arco_era5_us_airport_hourly
Airport count: 25126
Expected shards per month: 6
Expected month partitions: 120


In [ ]:
# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def month_dir(year, month):
    return os.path.join(ARCO_ROOT, f"year={year}", f"month={month:02d}")


def success_path(year, month):
    return os.path.join(month_dir(year, month), "_SUCCESS.json")


def monthly_manifest_path(year, month):
    return os.path.join(
        MANIFEST_ROOT,
        f"arco_weather_manifest_year={year}_month={month:02d}.json",
    )


def expected_hours_for_month(year, month):
    start = pd.Timestamp(year=year, month=month, day=1)

    if month == 12:
        end = pd.Timestamp(year=year + 1, month=1, day=1)
    else:
        end = pd.Timestamp(year=year, month=month + 1, day=1)

    return int((end - start).total_seconds() // 3600)


def list_month_parquet_files(year, month):
    base = month_dir(year, month)

    parquet_files = []

    if not os.path.exists(base):
        return parquet_files

    for root, dirs, files in os.walk(base):
        for file in files:
            if file.endswith(".parquet"):
                parquet_files.append(os.path.join(root, file))

    return sorted(parquet_files)


def read_json_if_exists(path):
    if not os.path.exists(path):
        return None

    with open(path, "r") as f:
        return json.load(f)


def parquet_row_count(path):
    pf = pq.ParquetFile(path)
    return pf.metadata.num_rows


def parquet_schema_names(path):
    pf = pq.ParquetFile(path)
    return pf.schema.names

In [ ]:
# ---------------------------------------------------------------------
# Main verification scan
# ---------------------------------------------------------------------

verification_rows = []
schema_problems = []
missing_months = []
missing_success = []
missing_manifests = []
row_count_mismatches = []
shard_count_mismatches = []
missing_files_list = []

for year in EXPECTED_YEARS:
    for month in EXPECTED_MONTHS:
        base = month_dir(year, month)
        success = success_path(year, month)
        manifest_file = monthly_manifest_path(year, month)

        exists_month_dir = os.path.isdir(base)
        exists_success = os.path.exists(success)
        exists_manifest = os.path.exists(manifest_file)

        if not exists_month_dir:
            missing_months.append((year, month))

        if not exists_success:
            missing_success.append((year, month))

        if not exists_manifest:
            missing_manifests.append((year, month))

        parquet_files = list_month_parquet_files(year, month)
        parquet_file_count = len(parquet_files)

        expected_hours = expected_hours_for_month(year, month)
        expected_rows = expected_hours * airport_count

        actual_rows = 0
        total_size_mb = 0.0

        for file_path in parquet_files:
            try:
                actual_rows += parquet_row_count(file_path)
                total_size_mb += os.path.getsize(file_path) / (1024 * 1024)

                schema_names = parquet_schema_names(file_path)
                missing_vars = [v for v in EXPECTED_VARIABLES if v not in schema_names]

                required_base_cols = ["time_utc", "airport_key", "day", "hour_utc"]
                missing_base_cols = [c for c in required_base_cols if c not in schema_names]

                if missing_vars or missing_base_cols:
                    schema_problems.append(
                        {
                            "year": year,
                            "month": month,
                            "file": file_path,
                            "missing_base_cols": missing_base_cols,
                            "missing_vars": missing_vars,
                        }
                    )

            except Exception as e:
                missing_files_list.append(
                    {
                        "year": year,
                        "month": month,
                        "file": file_path,
                        "error": repr(e),
                    }
                )

        success_json = read_json_if_exists(success)
        manifest_json = read_json_if_exists(manifest_file)

        success_rows = success_json.get("rows") if success_json else None
        manifest_rows = manifest_json.get("rows") if manifest_json else None

        success_file_count = success_json.get("file_count") if success_json else None
        manifest_file_count = manifest_json.get("file_count") if manifest_json else None

        status = "OK"

        if not exists_month_dir:
            status = "MISSING_MONTH_DIR"
        elif not exists_success:
            status = "MISSING_SUCCESS"
        elif parquet_file_count == 0:
            status = "NO_PARQUET_FILES"
        elif parquet_file_count != expected_shards_per_month:
            status = "SHARD_COUNT_MISMATCH"
            shard_count_mismatches.append((year, month, parquet_file_count, expected_shards_per_month))
        elif actual_rows != expected_rows:
            status = "ROW_COUNT_MISMATCH"
            row_count_mismatches.append((year, month, actual_rows, expected_rows))
        elif success_rows is not None and success_rows != expected_rows:
            status = "SUCCESS_ROW_MISMATCH"
            row_count_mismatches.append((year, month, success_rows, expected_rows))
        elif manifest_rows is not None and manifest_rows != expected_rows:
            status = "MANIFEST_ROW_MISMATCH"
            row_count_mismatches.append((year, month, manifest_rows, expected_rows))

        verification_rows.append(
            {
                "year": year,
                "month": month,
                "status": status,
                "month_dir_exists": exists_month_dir,
                "success_exists": exists_success,
                "manifest_exists": exists_manifest,
                "parquet_files": parquet_file_count,
                "expected_shards": expected_shards_per_month,
                "actual_rows_from_parquet_metadata": actual_rows,
                "expected_rows": expected_rows,
                "success_rows": success_rows,
                "manifest_rows": manifest_rows,
                "size_mb": round(total_size_mb, 2),
                "path": base,
            }
        )

verification_df = pd.DataFrame(verification_rows)

display(verification_df)

,year,month,status,month_dir_exists,success_exists,manifest_exists,parquet_files,expected_shards,actual_rows_from_parquet_metadata,expected_rows,success_rows,manifest_rows,size_mb,path
0,2015,1,OK,True,True,True,6,6,18693744,18693744,18693744,18693744,379.44,/content/drive/MyDrive/aviation_weather_disrup...
1,2015,2,OK,True,True,True,6,6,16884672,16884672,16884672,16884672,349.42,/content/drive/MyDrive/aviation_weather_disrup...
2,2015,3,OK,True,True,True,6,6,18693744,18693744,18693744,18693744,378.60,/content/drive/MyDrive/aviation_weather_disrup...
3,2015,4,OK,True,True,True,6,6,18090720,18090720,18090720,18090720,379.51,/content/drive/MyDrive/aviation_weather_disrup...
4,2015,5,OK,True,True,True,6,6,18693744,18693744,18693744,18693744,390.06,/content/drive/MyDrive/aviation_weather_disrup...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2024,8,OK,True,True,True,6,6,18693744,18693744,18693744,18693744,386.84,/content/drive/MyDrive/aviation_weather_disrup...
116,2024,9,OK,True,True,True,6,6,18090720,18090720,18090720,18090720,365.53,/content/drive/MyDrive/aviation_weather_disrup...
117,2024,10,OK,True,True,True,6,6,18693744,18693744,18693744,18693744,364.23,/content/drive/MyDrive/aviation_weather_disrup...
118,2024,11,OK,True,True,True,6,6,18090720,18090720,18090720,18090720,373.84,/content/drive/MyDrive/aviation_weather_disrup...


In [ ]:
# ---------------------------------------------------------------------
# Summary tables
# ---------------------------------------------------------------------

summary = {
    "expected_months": len(EXPECTED_YEARS) * len(EXPECTED_MONTHS),
    "months_found": int(verification_df["month_dir_exists"].sum()),
    "success_markers_found": int(verification_df["success_exists"].sum()),
    "monthly_manifests_found": int(verification_df["manifest_exists"].sum()),
    "total_parquet_files": int(verification_df["parquet_files"].sum()),
    "expected_total_parquet_files": int(len(EXPECTED_YEARS) * len(EXPECTED_MONTHS) * expected_shards_per_month),
    "total_rows_from_parquet_metadata": int(verification_df["actual_rows_from_parquet_metadata"].sum()),
    "expected_total_rows": int(verification_df["expected_rows"].sum()),
    "total_size_gb": round(float(verification_df["size_mb"].sum()) / 1024, 2),
    "ok_months": int((verification_df["status"] == "OK").sum()),
    "problem_months": int((verification_df["status"] != "OK").sum()),
}

summary_df = pd.DataFrame([summary])
display(summary_df)

status_counts = verification_df["status"].value_counts().reset_index()
status_counts.columns = ["status", "month_count"]
display(status_counts)

year_summary = (
    verification_df
    .groupby("year")
    .agg(
        months=("month", "count"),
        ok_months=("status", lambda s: int((s == "OK").sum())),
        parquet_files=("parquet_files", "sum"),
        rows=("actual_rows_from_parquet_metadata", "sum"),
        expected_rows=("expected_rows", "sum"),
        size_mb=("size_mb", "sum"),
    )
    .reset_index()
)

year_summary["size_gb"] = year_summary["size_mb"] / 1024
display(year_summary)

,expected_months,months_found,success_markers_found,monthly_manifests_found,total_parquet_files,expected_total_parquet_files,total_rows_from_parquet_metadata,expected_total_rows,total_size_gb,ok_months,problem_months
0,120,120,120,120,720,720,2202846672,2202846672,44.44,120,0


,status,month_count
0,OK,120


,year,months,ok_months,parquet_files,rows,expected_rows,size_mb,size_gb
0,2015,12,12,72,220103760,220103760,4517.43,4.411553
1,2016,12,12,72,220706784,220706784,4556.81,4.450010
2,2017,12,12,72,220103760,220103760,4570.78,4.463652
3,2018,12,12,72,220103760,220103760,4575.01,4.467783
4,2019,12,12,72,220103760,220103760,4572.24,4.465078
5,2020,12,12,72,220706784,220706784,4555.36,4.448594
6,2021,12,12,72,220103760,220103760,4543.71,4.437217
7,2022,12,12,72,220103760,220103760,4552.02,4.445332
8,2023,12,12,72,220103760,220103760,4523.59,4.417568
9,2024,12,12,72,220706784,220706784,4544.58,4.438066


In [ ]:
# ---------------------------------------------------------------------
# Problem details, if any
# ---------------------------------------------------------------------

problem_df = verification_df[verification_df["status"] != "OK"].copy()

if len(problem_df) == 0:
    print("✅ All monthly partitions passed verification.")
else:
    print("❌ Some monthly partitions have problems.")
    display(problem_df)

if missing_months:
    print("Missing month directories:")
    print(missing_months)

if missing_success:
    print("Missing _SUCCESS.json markers:")
    print(missing_success)

if missing_manifests:
    print("Missing monthly manifest JSON files:")
    print(missing_manifests)

if shard_count_mismatches:
    print("Shard-count mismatches:")
    print(shard_count_mismatches)

if row_count_mismatches:
    print("Row-count mismatches:")
    print(row_count_mismatches)

if schema_problems:
    print("Schema problems found:", len(schema_problems))
    display(pd.DataFrame(schema_problems).head(20))

if missing_files_list:
    print("Unreadable parquet files:", len(missing_files_list))
    display(pd.DataFrame(missing_files_list).head(20))

✅ All monthly partitions passed verification.


In [ ]:
# ---------------------------------------------------------------------
# Final pass/fail assertion
# ---------------------------------------------------------------------

all_ok = (
    summary["problem_months"] == 0
    and summary["months_found"] == summary["expected_months"]
    and summary["success_markers_found"] == summary["expected_months"]
    and summary["total_rows_from_parquet_metadata"] == summary["expected_total_rows"]
    and len(schema_problems) == 0
    and len(missing_files_list) == 0
)

if all_ok:
    print("✅ DATASET VERIFICATION PASSED")
    print(f"Verified years: {EXPECTED_YEARS[0]}–{EXPECTED_YEARS[-1]}")
    print(f"Verified months: {summary['expected_months']}")
    print(f"Total parquet files: {summary['total_parquet_files']}")
    print(f"Total rows: {summary['total_rows_from_parquet_metadata']:,}")
    print(f"Total size: {summary['total_size_gb']} GB")
else:
    raise RuntimeError("❌ DATASET VERIFICATION FAILED. Check problem_df and summaries above.")

✅ DATASET VERIFICATION PASSED
Verified years: 2015–2024
Verified months: 120
Total parquet files: 720
Total rows: 2,202,846,672
Total size: 44.44 GB
